# V-shapes paper_live — SHADOW board

**Class:** risk-monitor shadow · `live_orders=false` · `alpha_claim=false`

MinV / EGARCH = **Promote Monitor**. Paper throttle = **Kill/do-not-size** (observation only). Never soft-Promote.

See [`README.md`](README.md) · [`../paper_throttle/EXP_REPORT.md`](../paper_throttle/EXP_REPORT.md) · [`../../DESK_MEMO.md`](../../DESK_MEMO.md).

In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, display

PKG = Path('.').resolve()
LOG = PKG / 'logs' / 'shadow.log'
EVENTS = PKG / 'out' / 'events.jsonl'
BOARD = PKG / 'out' / 'SHADOW_BOARD.md'
META = PKG / 'out' / 'shadow_meta.json'
print('PKG', PKG)
print('log exists', LOG.is_file(), LOG)
print('events exists', EVENTS.is_file(), EVENTS)

## Tail commands

```bash
tail -f ${ARES_MICROSTRUCTURE:-$HOME/srv/ares-microstructure}/research/books/v_shapes/applications/paper_live/logs/shadow.log
tail -f ${ARES_MICROSTRUCTURE:-$HOME/srv/ares-microstructure}/research/books/v_shapes/applications/paper_live/out/events.jsonl
```

Start (smoke / forever):

```bash
cd ${ARES_MICROSTRUCTURE:-$HOME/srv/ares-microstructure}/research/books/v_shapes/applications/paper_live
python3 run_paper_live.py --iterations 1
python3 run_paper_live.py --poll --interval 120
```

In [ ]:
# Last 40 log lines
if LOG.is_file():
    lines = LOG.read_text(errors='replace').splitlines()
    print('\n'.join(lines[-40:]))
else:
    print('no shadow.log yet — run: python3 run_paper_live.py --iterations 1')

In [ ]:
# Recent JSONL events — interpret gate labels
rows = []
if EVENTS.is_file():
    for line in EVENTS.read_text().splitlines()[-50:]:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
print('n_events_tail', len(rows))
for r in rows[-15:]:
    ev = r.get('event')
    gate = r.get('gate') or r.get('label')
    print(f"{r.get('ts_utc','?'):24} {ev:22} gate={gate} venue={r.get('venue')} day={r.get('day')}")
    if ev == 'hypothetical_throttle':
        h = r.get('hypothetical') or {}
        print(f"    → HYPOTHETICAL Kill/do-not-size widen={h.get('widen_bps')} size={h.get('size_mult')} pov={h.get('pov')} deploy={r.get('deploy')}")

## How to read the tags

| Log / event | Meaning |
|-------------|--------|
| `BREACH` / `event=breach` | MinV < EGARCH 5% — **Promote Monitor** (not a size signal) |
| `SCORE` / `event=score` | Calendar Ridge — **Promote Monitor only**; never sized |
| `HYPOTHETICAL` / `hypothetical_throttle` | Paper widen/cut/POV — **Kill/do-not-size**; observation only |
| `HEARTBEAT` | Poll summary; `live_orders=false` always |

If you see `deploy=false` + `Kill` — that is correct. Do not quote from this package.

In [ ]:
if BOARD.is_file():
    display(Markdown(BOARD.read_text()))
elif META.is_file():
    meta = json.loads(META.read_text())
    print(json.dumps({k: meta.get(k) for k in ('day','venue','symbol','any_breach','gates','live_orders')}, indent=2))
else:
    print('no board/meta yet')